In [0]:
# Paths used throughout this notebook
SOURCE_DIR = "/Volumes/healthcare_dev/landing/cms_downloads"
RAW_DIR = "/Volumes/healthcare_dev/landing/raw"

# Check that the uploaded files are visible
display(dbutils.fs.ls(SOURCE_DIR))

In [0]:
import zipfile
import os

EXTRACT_DIR = f"{SOURCE_DIR}/extracted"
os.makedirs(EXTRACT_DIR, exist_ok=True)

zip_files = ["All FFS Claims.zip", "All Beneficiary Years.zip"]

for zip_name in zip_files:
    with zipfile.ZipFile(f"{SOURCE_DIR}/{zip_name}") as z:
        z.extractall(EXTRACT_DIR)
        print(f"{zip_name}: extracted {len(z.namelist())} files")

display(dbutils.fs.ls(EXTRACT_DIR))

In [0]:
from pyspark.sql import functions as F

carrier = (
    spark.read
    .option("header", True)
    .option("sep", "|")
    .csv(f"{EXTRACT_DIR}/carrier.csv")
)

print(f"Total rows: {carrier.count():,}")

# Count rows per year, using the claim thru date
display(
    carrier
    .withColumn("thru_year", F.year(F.to_date("CLM_THRU_DT", "dd-MMM-yyyy")))
    .groupBy("thru_year")
    .count()
    .orderBy("thru_year")
)

In [0]:
import os

BATCH_DIR = f"{SOURCE_DIR}/batches"

def split_by_year(file_name, date_col, file_type, src_dir=EXTRACT_DIR):
    """Split a pipe-delimited CMS file into one file per year of date_col.
    Lines are copied unchanged. Returns {year: row_count}."""
    src = f"{src_dir}/{file_name}"
    outputs, counts = {}, {}

    with open(src, "r", encoding="utf-8") as f:
        header = f.readline()
        date_idx = header.rstrip("\r\n").split("|").index(date_col)

        for line in f:
            date_value = line.split("|")[date_idx]      # e.g. "25-Mar-2015"
            year = date_value[-4:]
            if not year.isdigit():
                raise ValueError(f"Bad date '{date_value}' in {file_name}")

            if year not in outputs:
                folder = f"{BATCH_DIR}/{file_type}/batch_{year}"
                os.makedirs(folder, exist_ok=True)
                outputs[year] = open(f"{folder}/{file_type}.csv", "w", encoding="utf-8")
                outputs[year].write(header)
                counts[year] = 0

            outputs[year].write(line)
            counts[year] += 1

    for out in outputs.values():
        out.close()
    return dict(sorted(counts.items()))

# Test on carrier (already done, so commented out)
# carrier_counts = split_by_year("carrier.csv", "CLM_THRU_DT", "carrier")
# print(carrier_counts)
# print(f"Total: {sum(carrier_counts.values()):,}")

In [0]:
claim_files = ["inpatient", "outpatient", "snf", "hha", "hospice", "dme"]

for file_type in claim_files:
    # Independent check: count data rows in the original file (minus the header)
    with open(f"{EXTRACT_DIR}/{file_type}.csv", "r", encoding="utf-8") as f:
        expected = sum(1 for _ in f) - 1

    counts = split_by_year(f"{file_type}.csv", "CLM_THRU_DT", file_type)
    total = sum(counts.values())
    status = "OK" if total == expected else "MISMATCH"

    print(f"{file_type:<11} total={total:>8,}  expected={expected:>8,}  {status}  years={list(counts)}")

In [0]:
import shutil

# --- Part D: split by fill date ---
with open(f"{SOURCE_DIR}/pde.csv", "r", encoding="utf-8") as f:
    expected = sum(1 for _ in f) - 1

pde_counts = split_by_year("pde.csv", "SRVC_DT", "pde", src_dir=SOURCE_DIR)
total = sum(pde_counts.values())
print(f"pde  total={total:,}  expected={expected:,}  {'OK' if total == expected else 'MISMATCH'}  years={list(pde_counts)}")

# --- Beneficiary: already one file per year, so just copy ---
for year in range(2015, 2026):
    name = f"beneficiary_{year}.csv"
    folder = f"{BATCH_DIR}/beneficiary/batch_{year}"
    os.makedirs(folder, exist_ok=True)

    src = f"{EXTRACT_DIR}/{name}"
    dst = f"{folder}/{name}"
    shutil.copyfile(src, dst)

    same_size = os.path.getsize(src) == os.path.getsize(dst)
    print(f"{name}  {'OK' if same_size else 'MISMATCH'}")